# 04 - Parallelle Meervoudige Tool Uitvoering (Nederlandse Editie)

Deze notebook demonstreert geavanceerde parallelle tool-uitvoering met Azure OpenAI, met praktijkvoorbeelden specifiek voor Nederland.

## 🇳🇱 Nederlandse Context
We gebruiken realistische Nederlandse scenario's:
- Reisplanning tussen Nederlandse steden
- Weer vergelijkingen voor meerdere locaties
- NS en Schiphol informatie combineren

## 🎯 Leerdoelen
- Parallelle tool-aanroepen begrijpen en implementeren
- Efficiënte multi-tool workflows bouwen
- Token-gebruik optimaliseren

In [ ]:
import os
import json
from datetime import datetime
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Azure OpenAI Configuratie
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview"
)

deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client geïnitialiseerd voor parallelle uitvoering")

## 🔧 Definieer Nederlandse Tools voor Parallelle Uitvoering

In [ ]:
# Uitgebreide Nederlandse tools
parallel_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_dutch_weather",
            "description": "Verkrijg het huidige weer voor een Nederlandse stad",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "De Nederlandse stad"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_ns_connection",
            "description": "Verkrijg NS treinverbinding tussen twee stations",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_station": {"type": "string"},
                    "to_station": {"type": "string"}
                },
                "required": ["from_station", "to_station"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_city_info",
            "description": "Verkrijg informatie over een Nederlandse stad",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "De stad"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_events_today",
            "description": "Verkrijg evenementen vandaag in een stad",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    }
]

print(f"🔧 {len(parallel_tools)} tools gedefinieerd voor parallelle uitvoering")

In [ ]:
# Implementeer de tool functies
def get_dutch_weather(city: str) -> dict:
    weather_data = {
        "amsterdam": {"temp": 15, "condition": "Bewolkt", "rain_chance": "60%"},
        "rotterdam": {"temp": 16, "condition": "Gedeeltelijk bewolkt", "rain_chance": "40%"},
        "utrecht": {"temp": 14, "condition": "Regen", "rain_chance": "80%"},
        "den haag": {"temp": 15, "condition": "Zonnig", "rain_chance": "20%"},
        "eindhoven": {"temp": 17, "condition": "Helder", "rain_chance": "10%"}
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "Onbekend", "rain_chance": "50%"})
    return {"city": city, "temperature": f"{data['temp']}°C", "condition": data["condition"], "rain_chance": data["rain_chance"]}

def get_ns_connection(from_station: str, to_station: str) -> dict:
    return {
        "from": from_station,
        "to": to_station,
        "duration": "45 minuten",
        "price": "€15,80",
        "next_departure": "Over 12 minuten",
        "platform": "Spoor 5b"
    }

def get_city_info(city: str) -> dict:
    city_data = {
        "amsterdam": {"population": "872.000", "province": "Noord-Holland", "attractions": ["Rijksmuseum", "Anne Frank Huis", "Vondelpark"]},
        "rotterdam": {"population": "651.000", "province": "Zuid-Holland", "attractions": ["Erasmusbrug", "Markthal", "Euromast"]},
        "utrecht": {"population": "361.000", "province": "Utrecht", "attractions": ["Dom Tower", "Oudegracht", "Rietveld Schröderhuis"]}
    }
    data = city_data.get(city.lower(), {"population": "Onbekend", "province": "Onbekend", "attractions": []})
    return {"city": city, **data}

def get_events_today(city: str) -> dict:
    events = {
        "amsterdam": [{"name": "Concertgebouw: Klassiek concert", "time": "20:00"}, {"name": "AFAS Live: Pop concert", "time": "21:00"}],
        "rotterdam": [{"name": "Ahoy: Comedy show", "time": "20:30"}],
        "utrecht": [{"name": "TivoliVredenburg: Jazz avond", "time": "19:30"}]
    }
    return {"city": city, "events": events.get(city.lower(), [])}

def execute_tool(name: str, args: dict) -> str:
    tools = {
        "get_dutch_weather": get_dutch_weather,
        "get_ns_connection": get_ns_connection,
        "get_city_info": get_city_info,
        "get_events_today": get_events_today
    }
    return json.dumps(tools[name](**args), ensure_ascii=False)

print("✅ Tool functies geïmplementeerd")

## 🚀 Test Parallelle Uitvoering

In [ ]:
def process_parallel_query(query: str):
    """Verwerk een query met mogelijk parallelle tool-aanroepen"""
    
    print(f"❓ Query: {query}")
    print("=" * 60)
    
    messages = [{"role": "user", "content": query}]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        tools=parallel_tools,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    
    if assistant_message.tool_calls:
        print(f"\n🔧 {len(assistant_message.tool_calls)} PARALLELLE tool-aanroepen gedetecteerd!")
        
        messages.append(assistant_message)
        
        for i, tool_call in enumerate(assistant_message.tool_calls, 1):
            args = json.loads(tool_call.function.arguments)
            print(f"   [{i}] {tool_call.function.name}: {args}")
            
            result = execute_tool(tool_call.function.name, args)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        
        final = client.chat.completions.create(
            model=deployment,
            messages=messages
        )
        
        print("\n🤖 Antwoord:")
        print("-" * 40)
        print(final.choices[0].message.content)
    else:
        print(assistant_message.content)

# Test met query die meerdere tools triggert
process_parallel_query("Vergelijk het weer in Amsterdam, Rotterdam en Utrecht vandaag")

In [ ]:
# Test complexere query
print("\n" + "="*80 + "\n")
process_parallel_query("Ik wil van Amsterdam naar Rotterdam reizen. Vertel me over beide steden, het weer, en welke evenementen er vandaag zijn.")

## 📊 Voordelen van Parallelle Uitvoering

| Aspect | Sequentieel | Parallel |
|--------|------------|----------|
| API calls | Meerdere | 1 + 1 |
| Latentie | Hoger | Lager |
| Kosten | Meer tokens | Minder tokens |
| Complexiteit | Eenvoudig | Iets complexer |

## 🎓 Wat Je Hebt Geleerd

✅ **Parallelle Aanroepen**: Meerdere tools tegelijk uitvoeren  
✅ **Efficiëntie**: Token-gebruik optimaliseren  
✅ **Nederlandse Context**: Praktische voorbeelden met NS, weer, steden  

## 🚀 Volgende Stappen

- **05_assistants_api_with_vector_stores.nl.ipynb** - Document-gebaseerde intelligentie
- **06_complete_rag_pipeline_deep_dive.nl.ipynb** - Volledige RAG implementatie